# 📊 Customer Churn Prediction - Exploratory Data Analysis & Modeling
This notebook demonstrates the step-by-step process of building a machine learning model to predict customer churn using the IBM Telco dataset.

**Objective:** Identify customers who are highly likely to leave, and determine the key factors driving their decision.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="darkgrid")

## 1. Data Loading & Inspection

In [ ]:
# Load the dataset
df = pd.read_csv('../dataset/customer_churn.csv')
display(df.head())
print(f"Dataset Shape: {df.shape}")

In [ ]:
# Check for missing values and data types
df.info()

## 2. Data Cleaning
We noticed that 'TotalCharges' is an object type but it should be numeric. Let's fix that.

In [ ]:
# Convert TotalCharges to numeric, coercing errors to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill missing TotalCharges with 0 (since they correspond to customers with 0 tenure)
df['TotalCharges'].fillna(0, inplace=True)

# Drop customerID as it is not useful for prediction
if 'customerID' in df.columns:
    df.drop('customerID', axis=1, inplace=True)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Target Variable Distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Churn', palette='Set2')
plt.title('Customer Churn Distribution')
plt.show()

In [ ]:
# Numerical Features Distribution by Churn
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(data=df, x='tenure', hue='Churn', multiple='stack', ax=ax[0], palette='Set2')
ax[0].set_title('Tenure Distribution')

sns.histplot(data=df, x='MonthlyCharges', hue='Churn', multiple='stack', ax=ax[1], palette='Set2')
ax[1].set_title('Monthly Charges Distribution')
plt.show()

## 4. Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Separate target and features
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})

# Encode categorical features
categorical_cols = X.select_dtypes(include=['object']).columns
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# Scale numerical features
scaler = StandardScaler()
X_train[['tenure', 'MonthlyCharges', 'TotalCharges']] = scaler.fit_transform(X_train[['tenure', 'MonthlyCharges', 'TotalCharges']])
X_test[['tenure', 'MonthlyCharges', 'TotalCharges']] = scaler.transform(X_test[['tenure', 'MonthlyCharges', 'TotalCharges']])

## 5. Model Training (Gradient Boosting)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

# Initialize and train the model
model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

## 6. Model Evaluation

In [ ]:
# Print Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Calculate ROC-AUC Score
auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC Score: {auc:.4f}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix')
plt.grid(False)
plt.show()

## 7. Feature Importance
Understanding which features impact churn the most.

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance.head(10), x='Importance', y='Feature', palette='viridis')
plt.title('Top 10 Most Important Features')
plt.show()